# Risk model: what it is for, our baseline, and how to improve it

**Status: implemented and run.** Sections 1-5 define the problem and the candidate fixes. Section 6 runs them, section 7 adds a DCC-GARCH model, and section 8 gives the conclusion and recommendations. All results are in-sample on 1976-2014 and use the endpoint-sensitivity data (the only full-period run). Proposals are marked with their source: **[AFML]** for *Advances in Financial Machine Learning* (López de Prado, 2018) and **[mine]** for my own suggestion.

1. What a risk model tries to do
2. Our baseline
3. Where the baseline falls short
4. Proposed improvements
5. How we judge any change
6. Results
7. Multivariate GARCH (DCC-GARCH)
8. Conclusion and recommendations

Code: `src/risk_model/` (`rolling.py`, `daily.py`, `garch.py`, `rescale.py`), `src/backtest/cross_validation.py`, `scripts/risk_model_study.py`. Tests: `tests/risk_model/test_new_models.py`, `tests/backtest/test_cross_validation.py`.

**Cost revision status (26 October 2026):** configuration now uses 2 bps as primary and 0/1/2/5 bps scenarios. Saved numerical conclusions describe the earlier 5 bps experiment; the [controlled rerun is now complete](../reports/cost_revision_20261026/SUMMARY.md), with dated results below or in that report. Earlier sections retain their historical numerical results.

## 1. What a risk model tries to do

A portfolio is a bet on returns, but returns are hard to forecast and risk is much easier to forecast. A risk model forecasts **how much the portfolio's return will vary next period**, so that the portfolio can be sized to take the amount of risk we intend.

**Output.** A forecast of the covariance matrix `Q` of asset returns: each asset's volatility, and how assets move together. The forecast risk of a portfolio with weights `w` is `sqrt(w'Qw)`.

**What it is used for**
| Use | Example |
|---|---|
| Sizing | Hold less of a volatile market (inverse-volatility sizing). |
| Limits | Scale the whole portfolio down if forecast risk exceeds a ceiling. |
| Attribution | Show which markets or asset classes contribute the risk. |
| Optimisation input | Mean-variance or risk-parity weights (not used by this strategy). |

**What makes one good**
- **Unbiased (calibrated).** Realised risk matches forecast risk on average. We measure this as the standard deviation of `return / forecast volatility`, which should be 1.0.
- **Low noise.** Estimates should not jump around because of sampling error. Noise causes turnover and wrong position sizes.
- **Responsive.** Estimates should react when conditions really change, such as a crisis.
- **Valid.** `Q` must be positive definite (no portfolio can have zero or negative variance).

These goals pull against each other. Using more history lowers noise but reacts slowly. Fewer parameters lowers noise but can miss real structure. The central difficulty is that the number of covariance entries grows with the square of the number of assets, while the history we have does not.

## 2. Our baseline

**Strategy context.** A monthly trend strategy over futures. The alpha model sets the direction (long or short from the last 12 months of returns). The risk model sets the size and the limits.

**Data.** Monthly returns of 58 futures markets (commodities, equity indices, fixed income, currencies), January 1976 to December 2014, 468 months. Markets join over time: between 17 and 58 report a return in a given month. A market is only eligible after 36 consecutive months of returns. The March 1997 results rest on an unverified endpoint assumption (see `docs/DATA_STATUS.md`).

**How `Q` is built each month** (information through month t-1):
1. **Volatility.** Standard deviation of each market's last 36 monthly returns, times sqrt(12), floored at 2%. All 36 months have equal weight.
2. **Correlation.** Sample correlation of the same 36 months, then shrunk toward the identity: `R_shrunk = 0.5 R + 0.5 I`.
3. `Q = D R_shrunk D`, with `D` the diagonal matrix of volatilities.

**How `Q` is used.** Each market starts at `0.40 / (number of eligible markets x its volatility)`. The portfolio is scaled down if forecast volatility exceeds 10% or gross exposure exceeds 3x.

**Baseline performance** (trend strategy, 10% band, 5 bps cost, from the earlier run)
| | Trend | STATIC (always long) |
|---|---|---|
| Sharpe | 1.071 | 0.484 |
| Calibration (target 1.0) | **1.27** | **1.41** |

Realised risk is 27% (trend) and 41% (STATIC) above what the model forecast, so the 10% ceiling is not a real 10% ceiling.

## 3. Where the baseline falls short

**Hypotheses for the under-forecast**
| # | Hypothesis | Source |
|---|---|---|
| H1 | **Too few observations for too many markets.** 17-58 markets from 36 months. The book's rule of thumb is that a non-singular covariance needs at least N(N+1)/2 independent observations; a 58-market matrix has 1,711 distinct entries. | [AFML] §16.3 |
| H2 | **Shrinkage toward the identity understates correlation.** It makes the portfolio look more diversified than it is, most of all in crises when correlations jump. Trend positions in correlated markets often share a sign, which makes this worse. | [mine] |
| H3 | **Fat tails and crises.** The standard deviation of standardised returns is sensitive to a few extreme months. The model may be about right most of the time and badly wrong in a few episodes. | [mine]; [AFML] §14.7 on skew and kurtosis |
| H4 | **Non-stationarity.** Correlations and volatilities shift over four decades, so a 3-year average is stale after a regime change. | [AFML] Ch. 17 (structural breaks) |

These can partly be told apart: H1 and H2 point to the estimator, H3 and H4 to the data's behaviour, which no estimator fixes.

## 4. Proposed improvements

Each fix lists the hypothesis (H1-H4, section 3) it addresses. Effort is a rough guide.

| Idea | Addresses | Rationale | Source | Effort |
|---|---|---|---|---|
| **B2. Tune the shrinkage level out of sample.** Choose it with purged cross-validation rather than in-sample. | H2 (and H1, since shrinkage is what compensates for too few observations) | The 0.1 result is in-sample; its robustness is untested. | [AFML] Ch. 12 | Medium |
| **B3. Separate windows for volatility and correlation.** For example, a shorter window or faster decay for volatility (it is the quantity that changes quickly), and a long window for correlations (which need many observations). | H1 and H4 | Volatility and correlation have different noise and persistence. Earlier runs changed both together. | [mine] | Low |
| **B4. Use daily data for the estimates.** Compute volatility and correlation from daily returns aggregated to a monthly horizon. | H1 | About 21 times more observations per window. Caveat: markets close at different times, which biases daily cross-market correlations toward zero; this needs care (for example, use weekly returns or lagged-return corrections). The repo already has an EWMA daily-volatility model (`src/risk_model/ewma.py`). | [mine] | Medium-High |
| **C1. Calibration-based rescale of the risk ceiling (fallback).** Estimate the ratio of realised to forecast volatility on a trailing window and scale the ceiling or `Q` by it. Use only if calibration is still above target after B2-B4. | The symptom (under-forecast), whatever its cause (H1-H4). It is the only idea here that also covers H3 (fat tails) | Targets the gap directly and costs one scalar. It must be estimated with past data only, since a fixed 1.27 fitted on the full sample leaks the future. | [mine] | Medium |

## 5. How we judge any change

**Primary metrics.** Calibration (target 1.0), then trend Sharpe, turnover, cost, and worst drawdown. A change must improve calibration without worsening the rest.

## 6. Results

Every run uses the same pipeline as the baseline (v2 trend signal, 10% band, 5 bps cost, same eligibility rules and months). Only the risk model changes. **Calibration** is the standard deviation of gross return over forecast monthly volatility (1.0 is perfect). Tables show trend and STATIC side by side.

### Step 1. The baseline reproduces

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path[:0] = [str(ROOT), str(ROOT / "scripts")]

from risk_model_study import (
    compare_models,
    garch_model,
    load_study,
    mean_correlation,
    paired_sharpe_difference,
    rescaled_run,
    rolling_model,
    daily_model,
    separate_window_models,
    shrinkage_cpcv,
    summarize,
    variance_ratio,
    wide,
)

pd.options.display.float_format = "{:.3f}".format
study = load_study()
sample = study.inputs[2].loc[study.base["start"] : study.base["end"]]
print(f"{sample.shape[1]} markets, {len(sample)} months")

ledgers = {}  # every ledger run in this notebook, by label, for the comparisons in section 8
base_table, base_ledgers = compare_models(study, {"baseline (36m equal, shrink 0.5)": rolling_model(study)})
ledgers.update({("baseline", k): v for k, v in base_ledgers["baseline (36m equal, shrink 0.5)"].items()})
wide(base_table)

58 markets, 468 months


strategy                          trend                            \
                                 sharpe calibration gross_vol_ann   
model                                                               
baseline (36m equal, shrink 0.5)  1.071       1.272         0.104   

strategy                                                   static              \
                                 max_drawdown turnover_ann sharpe calibration   
model                                                                           
baseline (36m equal, shrink 0.5)       -0.145        7.409  0.484       1.406   

strategy                                                                  
                                 gross_vol_ann max_drawdown turnover_ann  
model                                                                     
baseline (36m equal, shrink 0.5)         0.119       -0.559        0.725

Trend Sharpe 1.071 and calibration 1.27, STATIC 0.484 and 1.41, as in the earlier runs. The baseline is the reference for every comparison below.

### Step 2. B3: separate weighting for volatility and correlation
Volatility and correlation get different decay weights (half-lives in months) within the same 36 months. "Equal" is the baseline weighting. The universe cannot be lengthened beyond 36 months here, because a longer correlation window would change which markets are eligible and confound the comparison.

In [2]:
b3_table, b3_ledgers = compare_models(study, separate_window_models(study))
ledgers.update({("b3 " + m, s): l for m, d in b3_ledgers.items() for s, l in d.items()})
wide(b3_table)

strategy                  trend                                         \
                         sharpe calibration gross_vol_ann max_drawdown   
model                                                                    
equal / equal (baseline)  1.071       1.272         0.104       -0.145   
vol equal / corr hl24     1.075       1.258         0.104       -0.142   
vol equal / corr hl12     1.078       1.255         0.104       -0.140   
vol equal / corr hl6      1.081       1.268         0.104       -0.139   
vol hl12 / corr equal     1.090       1.276         0.104       -0.154   
vol hl12 / corr hl24      1.094       1.263         0.103       -0.154   
vol hl12 / corr hl12      1.096       1.260         0.103       -0.154   
vol hl12 / corr hl6       1.099       1.275         0.103       -0.153   
vol hl6 / corr equal      1.109       1.311         0.106       -0.163   
vol hl6 / corr hl24       1.112       1.298         0.105       -0.163   
vol hl6 / corr hl12       1.116       1.296         0.105       -0.163   
vol hl6 / corr hl6        1.118       1.312         0.105       -0.164   

strategy                              static                            \
                         turnover_ann sharpe calibration gross_vol_ann   
model                                                                    
equal / equal (baseline)        7.409  0.484       1.406         0.119   
vol equal / corr hl24           7.402  0.483       1.404         0.118   
vol equal / corr hl12           7.402  0.482       1.407         0.118   
vol equal / corr hl6            7.419  0.484       1.419         0.118   
vol hl12 / corr equal           7.526  0.479       1.398         0.118   
vol hl12 / corr hl24            7.517  0.477       1.398         0.118   
vol hl12 / corr hl12            7.516  0.476       1.402         0.118   
vol hl12 / corr hl6             7.534  0.479       1.417         0.118   
vol hl6 / corr equal            7.796  0.460       1.425         0.120   
vol hl6 / corr hl24             7.786  0.458       1.426         0.120   
vol hl6 / corr hl12             7.786  0.458       1.431         0.120   
vol hl6 / corr hl6              7.808  0.461       1.447         0.120   

strategy                                            
                         max_drawdown turnover_ann  
model                                               
equal / equal (baseline)       -0.559        0.725  
vol equal / corr hl24          -0.559        0.729  
vol equal / corr hl12          -0.559        0.739  
vol equal / corr hl6           -0.556        0.774  
vol hl12 / corr equal          -0.549        0.808  
vol hl12 / corr hl24           -0.550        0.820  
vol hl12 / corr hl12           -0.548        0.839  
vol hl12 / corr hl6            -0.543        0.886  
vol hl6 / corr equal           -0.563        1.050  
vol hl6 / corr hl24            -0.562        1.066  
vol hl6 / corr hl12            -0.561        1.089  
vol hl6 / corr hl6             -0.557        1.135

**Reading.** No combination moves calibration meaningfully (trend 1.26 to 1.31, STATIC 1.40 to 1.45). A fast volatility estimate raises trend Sharpe by 0.02 to 0.05 and deepens the worst drawdown from about -14.5% to as much as -16.4%, with higher turnover. Changing the correlation weighting alone does almost nothing. B3 does not address the under-forecast.

### Step 3. B2: shrinkage level, tuned out of sample
First the shrinkage grid. A smaller number means less shrinkage toward the identity (0.5 is the baseline). Positive definiteness holds for any value above zero.

In [3]:
GRID = [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1, 0.05, 0.02]
b2_table, b2_ledgers = compare_models(study, {s: rolling_model(study, shrinkage=s) for s in GRID})
ledgers.update({(f"s={m}", s): l for m, d in b2_ledgers.items() for s, l in d.items()})
b2_table["model"] = b2_table["model"].map(lambda s: f"shrinkage {s}")
wide(b2_table)

strategy        trend                                                      \
               sharpe calibration gross_vol_ann max_drawdown turnover_ann   
model                                                                       
shrinkage 0.5   1.071       1.272         0.104       -0.145        7.409   
shrinkage 0.4   1.073       1.218         0.102       -0.135        7.379   
shrinkage 0.3   1.076       1.171         0.101       -0.131        7.349   
shrinkage 0.25  1.078       1.150         0.100       -0.129        7.334   
shrinkage 0.2   1.080       1.130         0.099       -0.127        7.318   
shrinkage 0.15  1.082       1.112         0.098       -0.127        7.302   
shrinkage 0.1   1.085       1.095         0.097       -0.128        7.285   
shrinkage 0.05  1.087       1.079         0.096       -0.128        7.271   
shrinkage 0.02  1.088       1.070         0.096       -0.129        7.262   

strategy       static                                                      
               sharpe calibration gross_vol_ann max_drawdown turnover_ann  
model                                                                      
shrinkage 0.5   0.484       1.406         0.119       -0.559        0.725  
shrinkage 0.4   0.483       1.325         0.114       -0.536        0.719  
shrinkage 0.3   0.484       1.257         0.111       -0.516        0.706  
shrinkage 0.25  0.485       1.227         0.109       -0.506        0.700  
shrinkage 0.2   0.485       1.199         0.107       -0.497        0.693  
shrinkage 0.15  0.486       1.172         0.106       -0.489        0.685  
shrinkage 0.1   0.487       1.148         0.105       -0.481        0.677  
shrinkage 0.05  0.487       1.125         0.103       -0.473        0.669  
shrinkage 0.02  0.489       1.111         0.103       -0.469        0.665

Calibration falls steadily as shrinkage falls, and trend Sharpe, drawdown and turnover improve slightly at the same time. There is no interior optimum: the best value in the grid is the smallest.

Now the out-of-sample check, combinatorial purged cross-validation [AFML §12.4]: 8 contiguous groups, 2 held out per split (28 splits, 7 backtest paths), with 36 months dropped on each side of the test groups. Each split picks the shrinkage whose **training** calibration is closest to 1, then scores it on the held-out months.

In [4]:
rows = {}
for strategy in ["trend", "static"]:
    splits, paths = shrinkage_cpcv({s: b2_ledgers[s][strategy] for s in GRID}, baseline=0.5)
    rows[strategy] = {
        "splits": len(splits),
        "most common choice": splits.chosen.mode().iloc[0],
        "share of splits choosing it": (splits.chosen == splits.chosen.mode().iloc[0]).mean(),
        "test calibration, chosen (mean over splits)": splits.calibration.mean(),
        "test calibration, baseline 0.5 (mean)": splits.baseline_calibration.mean(),
        "splits where chosen is closer to 1": (
            (splits.calibration - 1).abs() < (splits.baseline_calibration - 1).abs()
        ).mean(),
        "test Sharpe, chosen (mean)": splits.sharpe.mean(),
        "test Sharpe, baseline (mean)": splits.baseline_sharpe.mean(),
        "paths: calibration (range)": f"{paths.calibration.min():.3f} to {paths.calibration.max():.3f}",
        "paths: Sharpe (range)": f"{paths.sharpe.min():.3f} to {paths.sharpe.max():.3f}",
    }
pd.DataFrame(rows)

,trend,static
splits,28,28
most common choice,0.020,0.020
share of splits choosing it,0.964,1.000
"test calibration, chosen (mean over splits)",1.066,1.100
"test calibration, baseline 0.5 (mean)",1.267,1.390
splits where chosen is closer to 1,0.893,1.000
"test Sharpe, chosen (mean)",1.102,0.511
"test Sharpe, baseline (mean)",1.086,0.506
paths: calibration (range),1.070 to 1.072,1.111 to 1.111
paths: Sharpe (range),1.087 to 1.088,0.489 to 0.489


**Reading.** The procedure picks the lowest shrinkage in the grid in almost every split, because the criterion improves monotonically as shrinkage falls. On held-out months the chosen value brings calibration closer to 1 than the baseline in 89% of splits for trend and all splits for STATIC, and the stitched paths give calibration of about 1.07 (trend) and 1.11 (STATIC), against 1.27 and 1.41. Mean trend Sharpe on held-out months is about 0.02 higher (1.10 vs 1.09). This is the one change that is clearly consistent across held-out periods. Because the choice sits on the edge of the grid, I do not read the exact value as meaningful; the finding is the direction (much less shrinkage than 0.5).

### Step 4. B4: daily data
First the data check. The 62 daily files cover 58 markets used here plus 4 not in the monthly data. As a rough check for contract-roll jumps, only 63 daily moves exceed 8 trailing standard deviations in 46 years across all markets. They are not spread evenly over the calendar (11 fall on the 27th of a month, which I did not investigate), so this is not a full roll audit, but I found no clear sign of roll jumps. Market RL has missing daily prices, which removes 72 market-months (0.4%) from the daily universe.

The variants below keep everything else fixed (36-month window, equal weights, same shrinkage, floor and universe), so the difference isolates daily returns. "Weekly corr" estimates correlation from 5-day sums to reduce the bias from markets that close at different times.

In [5]:
b4_models = {
    "monthly baseline, shrink 0.5": rolling_model(study),
    "daily vol + daily corr, shrink 0.5": daily_model(study, 1, 0.5),
    "daily vol + weekly corr, shrink 0.5": daily_model(study, 5, 0.5),
    "monthly, shrink 0.1": rolling_model(study, shrinkage=0.1),
    "daily vol + daily corr, shrink 0.1": daily_model(study, 1, 0.1),
    "daily vol + weekly corr, shrink 0.1": daily_model(study, 5, 0.1),
}
b4_table, b4_ledgers = compare_models(study, b4_models)
ledgers.update({("b4 " + m, s): l for m, d in b4_ledgers.items() for s, l in d.items()})
wide(b4_table)

strategy                             trend                            \
                                    sharpe calibration gross_vol_ann   
model                                                                  
monthly baseline, shrink 0.5         1.071       1.272         0.104   
daily vol + daily corr, shrink 0.5   1.048       1.368         0.109   
daily vol + weekly corr, shrink 0.5  1.048       1.332         0.109   
monthly, shrink 0.1                  1.085       1.095         0.097   
daily vol + daily corr, shrink 0.1   1.055       1.202         0.104   
daily vol + weekly corr, shrink 0.1  1.051       1.164         0.103   

strategy                                                      static  \
                                    max_drawdown turnover_ann sharpe   
model                                                                  
monthly baseline, shrink 0.5              -0.145        7.409  0.484   
daily vol + daily corr, shrink 0.5        -0.160        7.596  0.468   
daily vol + weekly corr, shrink 0.5       -0.161        7.581  0.467   
monthly, shrink 0.1                       -0.128        7.285  0.487   
daily vol + daily corr, shrink 0.1        -0.152        7.546  0.468   
daily vol + weekly corr, shrink 0.1       -0.148        7.507  0.469   

strategy                                                                    \
                                    calibration gross_vol_ann max_drawdown   
model                                                                        
monthly baseline, shrink 0.5              1.406         0.119       -0.559   
daily vol + daily corr, shrink 0.5        1.444         0.123       -0.586   
daily vol + weekly corr, shrink 0.5       1.412         0.122       -0.579   
monthly, shrink 0.1                       1.148         0.105       -0.481   
daily vol + daily corr, shrink 0.1        1.179         0.109       -0.505   
daily vol + weekly corr, shrink 0.1       1.149         0.108       -0.497   

strategy                                          
                                    turnover_ann  
model                                             
monthly baseline, shrink 0.5               0.725  
daily vol + daily corr, shrink 0.5         0.654  
daily vol + weekly corr, shrink 0.5        0.693  
monthly, shrink 0.1                        0.677  
daily vol + daily corr, shrink 0.1         0.610  
daily vol + weekly corr, shrink 0.1        0.661

In [6]:
vr = variance_ratio(study)
corr = mean_correlation(study)
pd.DataFrame(
    {
        "variance ratio, monthly / daily-implied (median across markets)": [vr.median()],
        "average pairwise correlation, monthly returns": [corr.monthly.mean()],
        "daily returns": [corr.daily.mean()],
        "weekly returns": [corr.weekly.mean()],
        "monthly, 2009-2014 windows": [corr.monthly.loc["2009":].mean()],
        "daily, 2009-2014 windows": [corr.daily.loc["2009":].mean()],
    },
    index=["value"],
).T

,value
"variance ratio, monthly / daily-implied (median across markets)",1.010
"average pairwise correlation, monthly returns",0.137
daily returns,0.123
weekly returns,0.131
"monthly, 2009-2014 windows",0.189
"daily, 2009-2014 windows",0.132


**Reading.** Daily data does not improve the forecast. At the baseline shrinkage, calibration is worse (trend 1.37 vs 1.27) and Sharpe is lower. It is not a volatility scaling problem: the ratio of monthly variance to the daily-implied variance is about 1, so scaling daily volatility up to a month is fair. The cause is correlation. Correlations from daily returns are lower than from monthly returns in the later windows (2009-2014 especially), consistent with markets closing at different times, so daily data makes the portfolio look more diversified. Weekly blocks help only a little. With less shrinkage (0.1), daily-based estimates are still no better than monthly ones. The extra observations do not buy a better risk forecast here.

### Step 5. Decision gate
B2 clearly helped, B3 and B4 did not. The combination to carry forward is low shrinkage on the monthly estimator. At shrinkage 0.02 calibration is 1.07 for trend (inside the 1.0 to 1.1 criterion) and 1.11 for STATIC (just outside). That is close but not on target, so step 6 is applied.

### Step 6. C1: calibration rescale
The covariance is scaled by the trailing 60-month realised-over-forecast ratio (at least 24 months, past months only, 1.0 before that), separately for trend and STATIC. Sizing is unchanged, so only the risk ceiling responds. This is an approximation: the ratio comes from the unscaled strategy's own past rather than the rescaled one's.

In [7]:
c1_rows = []
for label, shrinkage in [("shrink 0.5", 0.5), ("shrink 0.02", 0.02)]:
    model = rolling_model(study, shrinkage=shrinkage)
    plain = b2_ledgers[shrinkage]
    scaled = rescaled_run(study, model, plain)
    for strategy in plain:
        ledgers[(f"{label} + rescale", strategy)] = scaled[strategy]
        c1_rows.append({"model": label, "strategy": strategy, **summarize(plain[strategy])})
        c1_rows.append({"model": label + " + rescale", "strategy": strategy, **summarize(scaled[strategy])})
c1_table = pd.DataFrame(c1_rows)
wide(c1_table)

strategy               trend                                         \
                      sharpe calibration gross_vol_ann max_drawdown   
model                                                                 
shrink 0.5             1.071       1.272         0.104       -0.145   
shrink 0.5 + rescale   1.108       1.029         0.093       -0.124   
shrink 0.02            1.088       1.070         0.096       -0.129   
shrink 0.02 + rescale  1.110       1.012         0.090       -0.129   

strategy                           static                            \
                      turnover_ann sharpe calibration gross_vol_ann   
model                                                                 
shrink 0.5                   7.409  0.484       1.406         0.119   
shrink 0.5 + rescale         7.031  0.511       1.056         0.098   
shrink 0.02                  7.262  0.489       1.111         0.103   
shrink 0.02 + rescale        7.102  0.509       1.044         0.097   

strategy                                         
                      max_drawdown turnover_ann  
model                                            
shrink 0.5                  -0.559        0.725  
shrink 0.5 + rescale        -0.448        0.719  
shrink 0.02                 -0.469        0.665  
shrink 0.02 + rescale       -0.447        0.728

**Reading.** The rescale brings calibration to about 1.01 to 1.06 for both strategies and realised volatility close to the 10% ceiling (trend 9.0% to 9.3%). It also improves the other metrics: trend Sharpe rises from 1.071 to 1.108, the worst trend drawdown narrows from -14.5% to -12.4%, and the STATIC worst drawdown narrows from -56% to -45%. Combined with low shrinkage, calibration is 1.01 (trend) and 1.04 (STATIC). The drawdown improvement is likely because the ceiling binds harder after periods in which risk was under-forecast; I did not test this.

## 7. Multivariate GARCH (DCC-GARCH)

**Model.** Daily returns for each market follow a GARCH(1,1) variance with variance targeting; (alpha, beta) are pooled by asset class so markets with short histories get stable estimates. Correlations follow DCC (Engle, 2002) on the standardised residuals, with the two DCC parameters estimated by composite likelihood on 100 random market pairs (Engle, Shephard and Sheppard, 2008). Parameters are refit each January on data through the prior December, and the forecast for each month sums the 21 daily covariance forecasts of the next month. The result then gets the same correlation shrinkage, 2% volatility floor and universe as the other models. Everything is **[mine]** from the literature.

**Departures from the earlier design.** The likelihood is Gaussian quasi-likelihood, not Student-t, to keep the fit simple and fast. The DCC long-run correlation is the expanding sample correlation of the standardised residuals, not a 36-month window. Weekly blocks are not used for the GARCH correlations, so the non-synchronous closing bias found in step 4 applies here too.

**Controls.** "GARCH + constant corr" uses the GARCH volatility forecasts but holds correlation at its long-run value, so the difference from DCC isolates dynamic correlation. The daily-data comparison from step 4 isolates the effect of the data. This cell takes a few minutes (the models are fitted once and reused across variants).

In [8]:
garch = garch_model(study, 0.5)
garch_models = {
    "DCC-GARCH, shrink 0.5": garch,
    "DCC-GARCH, shrink 0.1": garch.variant(0.1),
    "GARCH + constant corr, shrink 0.5": garch.variant(0.5, False),
    "GARCH + constant corr, shrink 0.1": garch.variant(0.1, False),
}
g_table, g_ledgers = compare_models(study, garch_models)
ledgers.update({("garch " + m, s): l for m, d in g_ledgers.items() for s, l in d.items()})
reference = b4_table[b4_table.model.isin(["monthly baseline, shrink 0.5", "monthly, shrink 0.1", "daily vol + daily corr, shrink 0.1"])]
wide(pd.concat([reference, g_table]))

strategy                            trend                            \
                                   sharpe calibration gross_vol_ann   
model                                                                 
monthly baseline, shrink 0.5        1.071       1.272         0.104   
monthly, shrink 0.1                 1.085       1.095         0.097   
daily vol + daily corr, shrink 0.1  1.055       1.202         0.104   
DCC-GARCH, shrink 0.5               1.140       1.337         0.107   
DCC-GARCH, shrink 0.1               1.155       1.170         0.102   
GARCH + constant corr, shrink 0.5   1.130       1.406         0.109   
GARCH + constant corr, shrink 0.1   1.144       1.241         0.105   

strategy                                                     static  \
                                   max_drawdown turnover_ann sharpe   
model                                                                 
monthly baseline, shrink 0.5             -0.145        7.409  0.484   
monthly, shrink 0.1                      -0.128        7.285  0.487   
daily vol + daily corr, shrink 0.1       -0.152        7.546  0.468   
DCC-GARCH, shrink 0.5                    -0.154        9.226  0.449   
DCC-GARCH, shrink 0.1                    -0.141        9.130  0.470   
GARCH + constant corr, shrink 0.5        -0.158        9.276  0.436   
GARCH + constant corr, shrink 0.1        -0.148        9.198  0.458   

strategy                                                                   \
                                   calibration gross_vol_ann max_drawdown   
model                                                                       
monthly baseline, shrink 0.5             1.406         0.119       -0.559   
monthly, shrink 0.1                      1.148         0.105       -0.481   
daily vol + daily corr, shrink 0.1       1.179         0.109       -0.505   
DCC-GARCH, shrink 0.5                    1.454         0.125       -0.671   
DCC-GARCH, shrink 0.1                    1.185         0.110       -0.587   
GARCH + constant corr, shrink 0.5        1.474         0.125       -0.683   
GARCH + constant corr, shrink 0.1        1.201         0.112       -0.602   

strategy                                         
                                   turnover_ann  
model                                            
monthly baseline, shrink 0.5              0.725  
monthly, shrink 0.1                       0.677  
daily vol + daily corr, shrink 0.1        0.610  
DCC-GARCH, shrink 0.5                     2.137  
DCC-GARCH, shrink 0.1                     2.009  
GARCH + constant corr, shrink 0.5         2.066  
GARCH + constant corr, shrink 0.1         1.909

In [9]:
fitted = garch.fitted_parameters()
fitted[["dcc_a", "dcc_b", "COMM_alpha", "COMM_beta", "EQ_alpha", "EQ_beta", "FI_alpha", "FI_beta", "FX_alpha", "FX_beta"]].loc[[1984, 1996, 2008, 2014]]

,dcc_a,dcc_b,COMM_alpha,COMM_beta,EQ_alpha,EQ_beta,FI_alpha,FI_beta,FX_alpha,FX_beta
year,,,,,,,,,,
1984,0.012,0.982,0.075,0.918,0.052,0.947,0.068,0.931,0.080,0.909
1996,0.010,0.983,0.068,0.924,0.081,0.901,0.050,0.944,0.059,0.933
2008,0.012,0.984,0.060,0.933,0.072,0.919,0.038,0.959,0.053,0.944
2014,0.011,0.985,0.058,0.936,0.085,0.906,0.039,0.960,0.052,0.947


**Reading.**
- **Calibration is not fixed.** At shrinkage 0.5, DCC-GARCH gives 1.34 (trend) and 1.45 (STATIC), no better than the baseline. At shrinkage 0.1 it gives 1.17 and 1.19, worse than the simple monthly estimator at the same shrinkage (1.10 and 1.15).
- **Trend Sharpe is higher, at a price.** DCC-GARCH at shrinkage 0.1 has trend Sharpe about 1.15 against 1.09 for the monthly estimator at the same shrinkage, but turnover is 9.1 times per year against 7.3, and STATIC is worse (Sharpe 0.47, worst drawdown -59% vs -48%, turnover 2.0 vs 0.7). The trend gain is small next to the noise in a 39-year Sharpe (see section 8).
- **Dynamic correlation matters a little for trend.** DCC beats constant correlation by about 0.07 in trend calibration at both shrinkages, but only by about 0.02 for STATIC. That supports the idea that correlations move, though not enough to close the gap.
- **Fitted parameters are sensible and stable.** Persistence (alpha + beta) is about 0.98 to 0.999 in every asset class, and the DCC parameters are around a = 0.01, b = 0.98, so correlations move slowly.
- **Same limitation as daily data.** The correlation bias from non-synchronous closes is not addressed, which is a plausible reason the correlation dynamics do not help more.

### 7.1 DCC-GARCH with different shrinkage, with and without the rescale
The same shrinkage grid as the monthly estimator in step 3 (the GARCH models are fitted once and reused across values), then the C1 rescale on top. Each DCC-GARCH row is paired with the monthly estimator at the same shrinkage, so the comparison is like for like. The rescale uses the same rules as step 6 (trailing 60 months, at least 24, past only).

In [10]:
GARCH_GRID = [0.5, 0.25, 0.1, 0.05, 0.02]
dcc_models = {f"DCC-GARCH, shrink {s}": garch.variant(s) for s in GARCH_GRID}
dcc_table, dcc_ledgers = compare_models(study, dcc_models)
ledgers.update({(m, s): l for m, d in dcc_ledgers.items() for s, l in d.items()})

rows = []
for s in GARCH_GRID:
    for family, model, plain in [
        ("DCC-GARCH", dcc_models[f"DCC-GARCH, shrink {s}"], dcc_ledgers[f"DCC-GARCH, shrink {s}"]),
        ("monthly", rolling_model(study, shrinkage=s), b2_ledgers[s]),
    ]:
        scaled = rescaled_run(study, model, plain)
        for strategy in plain:
            ledgers[(f"{family} shrink {s} + rescale", strategy)] = scaled[strategy]
            ledgers.setdefault((f"{family} shrink {s}", strategy), plain[strategy])
            rows.append({"model": f"{family}, shrink {s}", "strategy": strategy, **summarize(plain[strategy])})
            rows.append({"model": f"{family}, shrink {s} + rescale", "strategy": strategy, **summarize(scaled[strategy])})
grid_table = pd.DataFrame(rows)
order = [f"{f}, shrink {s}{r}" for s in GARCH_GRID for f in ("DCC-GARCH", "monthly") for r in ("", " + rescale")]
wide(grid_table).loc[order]

strategy                          trend                            \
                                 sharpe calibration gross_vol_ann   
model                                                               
DCC-GARCH, shrink 0.5             1.140       1.337         0.107   
DCC-GARCH, shrink 0.5 + rescale   1.169       1.038         0.094   
monthly, shrink 0.5               1.071       1.272         0.104   
monthly, shrink 0.5 + rescale     1.108       1.029         0.093   
DCC-GARCH, shrink 0.25            1.148       1.222         0.104   
DCC-GARCH, shrink 0.25 + rescale  1.172       1.029         0.093   
monthly, shrink 0.25              1.078       1.150         0.100   
monthly, shrink 0.25 + rescale    1.110       1.019         0.091   
DCC-GARCH, shrink 0.1             1.155       1.170         0.102   
DCC-GARCH, shrink 0.1 + rescale   1.171       1.026         0.092   
monthly, shrink 0.1               1.085       1.095         0.097   
monthly, shrink 0.1 + rescale     1.110       1.014         0.091   
DCC-GARCH, shrink 0.05            1.157       1.154         0.101   
DCC-GARCH, shrink 0.05 + rescale  1.171       1.025         0.092   
monthly, shrink 0.05              1.087       1.079         0.096   
monthly, shrink 0.05 + rescale    1.110       1.013         0.090   
DCC-GARCH, shrink 0.02            1.159       1.145         0.101   
DCC-GARCH, shrink 0.02 + rescale  1.171       1.024         0.092   
monthly, shrink 0.02              1.088       1.070         0.096   
monthly, shrink 0.02 + rescale    1.110       1.012         0.090   

strategy                                                   static              \
                                 max_drawdown turnover_ann sharpe calibration   
model                                                                           
DCC-GARCH, shrink 0.5                  -0.154        9.226  0.449       1.454   
DCC-GARCH, shrink 0.5 + rescale        -0.111        8.635  0.515       1.069   
monthly, shrink 0.5                    -0.145        7.409  0.484       1.406   
monthly, shrink 0.5 + rescale          -0.124        7.031  0.511       1.056   
DCC-GARCH, shrink 0.25                 -0.146        9.170  0.465       1.267   
DCC-GARCH, shrink 0.25 + rescale       -0.115        8.658  0.514       1.055   
monthly, shrink 0.25                   -0.129        7.334  0.485       1.227   
monthly, shrink 0.25 + rescale         -0.126        7.076  0.510       1.048   
DCC-GARCH, shrink 0.1                  -0.141        9.130  0.470       1.185   
DCC-GARCH, shrink 0.1 + rescale        -0.117        8.671  0.513       1.049   
monthly, shrink 0.1                    -0.128        7.285  0.487       1.148   
monthly, shrink 0.1 + rescale          -0.128        7.096  0.510       1.045   
DCC-GARCH, shrink 0.05                 -0.140        9.116  0.472       1.161   
DCC-GARCH, shrink 0.05 + rescale       -0.118        8.677  0.513       1.047   
monthly, shrink 0.05                   -0.128        7.271  0.487       1.125   
monthly, shrink 0.05 + rescale         -0.128        7.100  0.510       1.044   
DCC-GARCH, shrink 0.02                 -0.139        9.109  0.474       1.147   
DCC-GARCH, shrink 0.02 + rescale       -0.119        8.680  0.512       1.047   
monthly, shrink 0.02                   -0.129        7.262  0.489       1.111   
monthly, shrink 0.02 + rescale         -0.129        7.102  0.509       1.044   

strategy                                                                  
                                 gross_vol_ann max_drawdown turnover_ann  
model                                                                     
DCC-GARCH, shrink 0.5                    0.125       -0.671        2.137  
DCC-GARCH, shrink 0.5 + rescale          0.099       -0.486        1.932  
monthly, shrink 0.5                      0.119       -0.559        0.725  
monthly, shrink 0.5 + rescale            0.098       -0.448        0.719  
DCC-GARCH, shrink 0.25 

In [11]:
trend = grid_table[grid_table.strategy == "trend"].set_index("model")
summary = pd.DataFrame(
    {
        "DCC-GARCH Sharpe": [trend.loc[f"DCC-GARCH, shrink {s} + rescale", "sharpe"] for s in GARCH_GRID],
        "monthly Sharpe": [trend.loc[f"monthly, shrink {s} + rescale", "sharpe"] for s in GARCH_GRID],
        "DCC-GARCH calibration": [trend.loc[f"DCC-GARCH, shrink {s} + rescale", "calibration"] for s in GARCH_GRID],
        "monthly calibration": [trend.loc[f"monthly, shrink {s} + rescale", "calibration"] for s in GARCH_GRID],
        "DCC-GARCH turnover": [trend.loc[f"DCC-GARCH, shrink {s} + rescale", "turnover_ann"] for s in GARCH_GRID],
        "monthly turnover": [trend.loc[f"monthly, shrink {s} + rescale", "turnover_ann"] for s in GARCH_GRID],
    },
    index=pd.Index(GARCH_GRID, name="shrinkage"),
)
summary.insert(2, "Sharpe gap (DCC - monthly), 90% interval", [
    "{difference:+.3f} [{low_5pct:+.3f}, {high_95pct:+.3f}]".format(**paired_sharpe_difference(
        ledgers[(f"DCC-GARCH shrink {s} + rescale", "trend")].net_return,
        ledgers[(f"monthly shrink {s} + rescale", "trend")].net_return,
    )) for s in GARCH_GRID
])
summary

,DCC-GARCH Sharpe,monthly Sharpe,"Sharpe gap (DCC - monthly), 90% interval",DCC-GARCH calibration,monthly calibration,DCC-GARCH turnover,monthly turnover
shrinkage,,,,,,,
0.500,1.169,1.108,"+0.061 [-0.008, +0.155]",1.038,1.029,8.635,7.031
0.250,1.172,1.110,"+0.062 [-0.010, +0.157]",1.029,1.019,8.658,7.076
0.100,1.171,1.110,"+0.061 [-0.011, +0.156]",1.026,1.014,8.671,7.096
0.050,1.171,1.110,"+0.061 [-0.011, +0.155]",1.025,1.013,8.677,7.100
0.020,1.171,1.110,"+0.061 [-0.011, +0.155]",1.024,1.012,8.680,7.102


**Reading.**
- **Once the rescale is on, shrinkage barely matters.** With the rescale, trend calibration is 1.01 to 1.04 and Sharpe is flat across the whole grid (monthly 1.108 to 1.110, DCC-GARCH 1.169 to 1.172), from heavy shrinkage (0.5) to almost none (0.02). The rescale absorbs the correlation bias that shrinkage causes. Lower shrinkage matters mainly when the rescale is not used.
- **The rescale works on DCC-GARCH too.** DCC-GARCH calibration falls from 1.34/1.45 (trend/STATIC) at shrinkage 0.5 to 1.04/1.07 with the rescale, and from 1.17/1.19 to 1.03/1.05 at shrinkage 0.1.
- **DCC-GARCH with the rescale has the best trend numbers, but the edge is borderline.** Trend Sharpe is 1.17 against 1.11 for the monthly estimator with the rescale. The paired 90% interval on the gap runs from about -0.01 to +0.16 at every shrinkage, so it touches zero. The worst trend drawdown is -11.7% against -12.8% (shrinkage 0.1). The cost is turnover of 8.7 times a year against 7.1.
- **STATIC gets no benefit from DCC-GARCH.** With the rescale, Sharpe is about the same (0.513 vs 0.510), the worst drawdown is slightly worse (-48% vs -45%), and turnover is 1.9 vs 0.7 times a year.
- **Caveat.** The rescale factor for each model comes from that model's unscaled past ratios (the approximation from step 6), and about 40 variants have now been tried on this sample.

## 8. Conclusion and recommendations

### What we learned

In [12]:
from itertools import chain

def trend(label):
    return ledgers[(label, "trend")].net_return

def pair(first, second, strategy="trend"):
    return paired_sharpe_difference(ledgers[(first, strategy)].net_return, ledgers[(second, strategy)].net_return)

comparisons = {
    "shrink 0.02 vs baseline": ("s=0.02", "baseline"),
    "shrink 0.5 + rescale vs baseline": ("shrink 0.5 + rescale", "baseline"),
    "shrink 0.02 + rescale vs shrink 0.02": ("shrink 0.02 + rescale", "s=0.02"),
    "separate windows (vol hl6 / corr hl6) vs baseline": ("b3 vol hl6 / corr hl6", "baseline"),
    "daily data vs monthly (shrink 0.1)": ("b4 daily vol + daily corr, shrink 0.1", "b4 monthly, shrink 0.1"),
    "DCC-GARCH vs monthly (shrink 0.1)": ("garch DCC-GARCH, shrink 0.1", "b4 monthly, shrink 0.1"),
}
out = {}
for name, (a, b) in comparisons.items():
    r = pair(a, b)
    out[name] = {"trend Sharpe difference": r["difference"], "90% interval low": r["low_5pct"], "90% interval high": r["high_95pct"]}
pd.DataFrame(out).T

,trend Sharpe difference,90% interval low,90% interval high
shrink 0.02 vs baseline,0.017,-0.010,0.042
shrink 0.5 + rescale vs baseline,0.037,-0.000,0.073
shrink 0.02 + rescale vs shrink 0.02,0.022,-0.001,0.044
separate windows (vol hl6 / corr hl6) vs baseline,0.047,0.002,0.096
daily data vs monthly (shrink 0.1),-0.030,-0.077,0.030
DCC-GARCH vs monthly (shrink 0.1),0.070,-0.004,0.170


The table is a paired moving-block bootstrap of the trend Sharpe difference (12-month blocks, 2,000 resamples), so each difference is judged against the noise in the difference itself. Only the separate-window variant (+0.047, interval 0.002 to 0.096) has an interval that excludes zero, and it comes with higher turnover and a deeper drawdown. The rescale (+0.037) and DCC-GARCH (+0.07) intervals touch zero, and the lower-shrinkage gain (+0.017) is within noise. About 40 variants were tried on this one sample, so even an interval that excludes zero is optimistic.

### Findings

| Idea | Calibration (trend / STATIC) | Verdict |
|---|---|---|
| Baseline (36m equal, shrink 0.5) | 1.27 / 1.41 | reference |
| B3 separate windows | 1.26 to 1.31 / 1.40 to 1.45 | no effect on calibration; small Sharpe gain with more turnover and a deeper drawdown |
| **B2 lower shrinkage (0.02)** | **1.07 / 1.11** | **clear improvement, consistent in held-out months** |
| B4 daily data | 1.37 / 1.44 at shrink 0.5 | no gain; daily correlations are biased low |
| **C1 rescale (on top of baseline)** | **1.03 / 1.06** | **fixes calibration; narrows drawdowns; Sharpe gain is borderline** |
| C1 rescale with shrink 0.02 | 1.01 / 1.04 | best calibration of everything tried |
| DCC-GARCH | 1.17 / 1.19 at shrink 0.1 | no calibration gain; higher turnover |
| DCC-GARCH + rescale | 1.03 / 1.05 at shrink 0.1 | best trend Sharpe (1.17) and trend drawdown (-11.7%), but the Sharpe gain over monthly + rescale is borderline and turnover is higher; STATIC no better |

**The cause.** The under-forecast is mostly a correlation-shrinkage effect (H2) and partly a persistent bias that a past-only scale factor can remove. The speed of the volatility estimate (H4 in its simplest form) is not the cause, and the extra observations from daily data (H1) did not help; daily correlations come out lower than monthly ones in later windows.

### Recommendations
1. **Reduce the correlation shrinkage from 0.5 to a low value.** This is the largest and most robust gain. Choose a value in the range 0.05 to 0.1 rather than the 0.02 edge of the grid, so the matrix stays well conditioned (with 17 to 58 markets and 36 months the sample correlation matrix is rank-deficient and shrinkage is what keeps it invertible). The out-of-sample check shows the direction, not the exact value.
2. **Add the calibration rescale (C1) as a second stage.** With the rescale on, the shrinkage level matters little (Sharpe is flat from 0.5 to 0.02), so if you adopt it, recommendation 1 becomes a refinement rather than a requirement. The rescale uses only past data, needs one scalar, closes the remaining gap, and lowers the worst drawdown (the Sharpe gain is borderline). Review it before adopting because it was implemented as an approximation (see caveats).
3. **Keep the monthly 36-month estimator.** Faster weighting (B3), daily data (B4) and DCC-GARCH all add complexity, turnover or both without fixing calibration. Their Sharpe gains are either borderline (DCC-GARCH, including with the rescale: +0.06, interval about -0.01 to +0.16, with 1.6 more turns a year) or bought with higher turnover and deeper drawdowns (B3). DCC-GARCH with the rescale is the most promising optional extension and should be re-tested on data outside 1976-2014 before adoption.
4. **Do not build further dynamic models until the correlation bias in daily data is addressed.** If revisited, use weekly returns or a lagged-return correction before fitting DCC.

### Caveats
- **Everything is in-sample on one 39-year history** with about 40 variants tried. Only the shrinkage result was checked with cross-validation, and the selection hit the edge of the grid. Data outside 1976-2014 would be the real test.
- **C1 approximation.** The scale factor for the rescaled strategy comes from the baseline ledger's past ratios, not the rescaled strategy's own. The rescaled strategy's calibration is measured correctly (it is about 1.01 to 1.06), so the approximation works, but a fully self-consistent version would need the rescale built into the simulation loop.
- **Endpoint assumption.** Results depend on the unverified March 1997 endpoint assumption (see `docs/DATA_STATUS.md`).
- **Not tested:** whether the remaining miscalibration is tail-driven (fat tails and crises, H3). A robust calibration measure or an exclusion of the worst months would separate that from a variance bias. Student-t GARCH, factor covariance models and nonlinear shrinkage were not tried.
- **Calibration target.** Trend and STATIC differ by 0.14 in calibration at the baseline. Whether to target 1.0 for both is a decision for you.

## Cost revision update — 26 October 2026

The earlier sections retain the 5 bps experiment. The following table comes from the separately identified `cost_revision_20261026` run at the same 10% band. No estimator has been promoted.

| Estimator | TREND Sharpe at 1 bps | TREND Sharpe at 2 bps | Calibration at 2 bps |
|---|---:|---:|---:|
| equal_36m | 1.099 | 1.092 | 1.272 |
| ewma_hl6 | 1.147 | 1.140 | 1.312 |
| ewma_hl12 | 1.125 | 1.118 | 1.260 |
| ewma_hl24 | 1.112 | 1.105 | 1.255 |
| fast_slow_3_equal | 1.144 | 1.136 | 1.288 |
| fast_slow_6_24 | 1.132 | 1.124 | 1.269 |

The calibration shortfall remains. These are exploratory, inspected historical comparisons, not independent validation. [Full table with STATIC](../reports/cost_revision_20261026/output/risk_model_comparison/cost_comparison.csv) · [Provenance and checks](../reports/cost_revision_20261026/SUMMARY.md).